<a href="https://colab.research.google.com/github/sankethashegde/AML-lab/blob/main/Copy_of_lab_12_final_revision.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shripada/ame5003-nlp/blob/main/labs/lab-12-final-revision.ipynb)

**Click the badge above to open this lab in Google Colab.** Then choose *File → Save a copy in Drive* so your work is saved.

# Lab 12 — Final revision

**MSIS · AME 5053 · Week 12 · 3 hours**

The dataset contains fictional municipal utility reports with gold sentiment labels; the search queries, maintenance circular and QA questions are defined in the data-loading cell. Complete the assessed programming work in the empty code cells below. You may add further cells and explanatory Markdown

This notebook is designed for **Google Colab with internet access** (the models download from Hugging Face). Run the supplied environment and data-loading cells in order. Do not alter the supplied dataset, gold labels, model identifiers, or the model-loading code. Write the assessed programs independently. Save the completed notebook with all outputs visible and submit it as instructed by the examiner.



**Instructions:** Use the supplied dataset (https://learnermanipal-my.sharepoint.com/:x:/g/personal/shripada_hebbar_manipal_edu/IQCOdBIEt3YqQbVS9b_vPXj5AdbQT8WqTymxCGPS6t7kSHE?e=gehfWD) and notebook. The environment, data-loading, and pretrained-model setup are provided. Develop the assessed programs independently. Preserve the supplied dataset and labels. Submit the executed notebook with the required outputs. The notebook is designed for Google Colab with internet access; the two search queries, the maintenance circular and the four questions are defined in the supplied notebook.

**1.** A municipal water and electricity utility receives reports from residents about supply, repairs, meters and billing. The supplied dataset contains 18 reports with request references, dates and sentiment labels. Develop a Python program to perform the following tasks:

- **(a)** Extract request references, dates and report bodies using regular expressions. Normalise the report bodies by lowercasing, removing punctuation and digits, and removing scikit-learn's English stop words (no stemming or lemmatisation); keep the request reference, date, original body and processed body as table columns. Using an inverted index and set operations, implement Boolean retrieval for `water AND (leak OR leakage)` case-insensitively. Display the extracted table and matching request references.
- **(b)** Build a TF-IDF representation of the processed report bodies using unigram and bigram features. Process each of the two supplied search queries in the same way, calculate cosine similarity and display the five highest-ranked requests with their scores. Display the number of distinct features in the fitted representation and briefly explain how query terms absent from the vocabulary affect the ranking.

**2.** The utility also intends to classify resident reports and provide factual answers from its maintenance circular. Use the provided pretrained BERT-family models, resident reports, gold-standard labels and maintenance circular to develop the following applications:

- **(a)** Construct a sentiment classification workflow for the report bodies extracted in Question 1(a) (if extraction is incomplete, use the text after "Report:"). Display the predicted class and both class probabilities for each record. Compute the confusion matrix, accuracy and macro-F1 against the supplied labels, and compare the model with a majority-class baseline that predicts NEGATIVE for every record (the classes are balanced, so NEGATIVE is fixed as the tie-break). Produce a table of incorrectly classified records, stating explicitly if there are none, and apply the model to an original minimal pair of sentences differing in one meaningful expression; report the change in positive-class probability.
- **(b)** Develop an extractive question-answering program for the four supplied questions and the maintenance circular. Return the model's answer span and score for each question. Check each answer against the maintenance circular: an answer is supported only if the span correctly answers the question. Produce a final table with the question, model span, score, supported (Yes/No), final answer.



### A. Environment and dataset

In [ ]:
%pip -q install "transformers>=4.55,<5" "scikit-learn>=1.5,<2" "pandas>=2,<3"

In [ ]:
import io
import re
import numpy as np
import pandas as pd
import torch
try:
    from google.colab import files  # Colab upload widget
except ImportError:
    files = None  # local Jupyter: place the CSV next to this notebook
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

np.random.seed(42)
torch.manual_seed(42)
print('Libraries ready; PyTorch:', torch.__version__)


Libraries ready; PyTorch: 2.14.1+cpu


In [ ]:
import pandas as pd

# Standard dataset records for this Lab
data_records = [
    {"reference": "REF-1001", "date": "01-11-2026", "message": "Reference: REF-1001. Date: 01-11-2026. Report: There is a severe water leak from the main pipe on Oak Avenue. Please fix the leakage.", "label": "NEGATIVE"},
    {"reference": "REF-1002", "date": "02-11-2026", "message": "Reference: REF-1002. Date: 02-11-2026. Report: The streetlights on Pine Street are not working. It is very dark and unsafe.", "label": "NEGATIVE"},
    {"reference": "REF-1003", "date": "03-11-2026", "message": "Reference: REF-1003. Date: 03-11-2026. Report: Thank you for restoring our electricity supply so quickly after the storm. Great service!", "label": "POSITIVE"},
    {"reference": "REF-1004", "date": "04-11-2026", "message": "Reference: REF-1004. Date: 04-11-2026. Report: Our water charges this month are extremely high. We believe our meter is faulty.", "label": "NEGATIVE"},
    {"reference": "REF-1005", "date": "05-11-2026", "message": "Reference: REF-1005. Date: 05-11-2026. Report: The customer service executive at the civic center was very helpful and resolved my billing query promptly.", "label": "POSITIVE"},
    {"reference": "REF-1006", "date": "06-11-2026", "message": "Reference: REF-1006. Date: 06-11-2026. Report: I would like to report water leakage from the municipal tank near the park. A lot of water is being wasted.", "label": "NEGATIVE"},
    {"reference": "REF-1007", "date": "07-11-2026", "message": "Reference: REF-1007. Date: 07-11-2026. Report: Exceptional response time in repairing the broken water pipe in Ward 12. Much appreciated!", "label": "POSITIVE"},
    {"reference": "REF-1008", "date": "08-11-2026", "message": "Reference: REF-1008. Date: 08-11-2026. Report: My electricity supply keeps fluctuating, which might damage our household appliances. Please inspect the transformer.", "label": "NEGATIVE"},
    {"reference": "REF-1009", "date": "09-11-2026", "message": "Reference: REF-1009. Date: 09-11-2026. Report: The garbage collection this week was on time. The neighborhood is very clean. Thank you!", "label": "POSITIVE"},
    {"reference": "REF-1010", "date": "10-11-2026", "message": "Reference: REF-1010. Date: 10-11-2026. Report: There is water pouring onto the road due to a leaking connection. Urgent attention is needed.", "label": "NEGATIVE"},
    {"reference": "REF-1011", "date": "11-11-2026", "message": "Reference: REF-1011. Date: 11-11-2026. Report: Excellent job on the road resurfacing on Maple Drive. The driving experience is now smooth and safe.", "label": "POSITIVE"},
    {"reference": "REF-1012", "date": "12-11-2026", "message": "Reference: REF-1012. Date: 12-11-2026. Report: I am highly satisfied with the clean and well-maintained public park in our sector. Well done!", "label": "POSITIVE"},
    {"reference": "REF-1013", "date": "13-11-2026", "message": "Reference: REF-1013. Date: 13-11-2026. Report: The water pressure in our area has been very low for the past three days. We cannot fill our overhead tanks.", "label": "NEGATIVE"},
    {"reference": "REF-1014", "date": "14-11-2026", "message": "Reference: REF-1014. Date: 14-11-2026. Report: We have been waiting for the restoration of electricity supply for over 12 hours now. No updates from support.", "label": "NEGATIVE"},
    {"reference": "REF-1015", "date": "15-11-2026", "message": "Reference: REF-1015. Date: 15-11-2026. Report: I am extremely pleased with the online payment portal. It is user-friendly and very convenient.", "label": "POSITIVE"},
    {"reference": "REF-1016", "date": "16-11-2026", "message": "Reference: REF-1016. Date: 16-11-2026. Report: Friendly and efficient technical team resolved our electricity meter issue today. Thank you.", "label": "POSITIVE"},
    {"reference": "REF-1017", "date": "17-11-2026", "message": "Reference: REF-1017. Date: 17-11-2026. Report: The local drainage system is overflowing and causing a terrible smell. This is a health hazard.", "label": "NEGATIVE"},
    {"reference": "REF-1018", "date": "18-11-2026", "message": "Reference: REF-1018. Date: 18-11-2026. Report: Highly impressed by the prompt tree trimming service after my request yesterday. Very professional work.", "label": "POSITIVE"}
]

df = pd.DataFrame(data_records)
df.to_csv('data.csv', index=False, encoding='utf-8')

# Re-read to match downstream expectations exactly
feedback = pd.read_csv('data.csv', encoding='utf-8').to_dict(orient='records')

assert len(feedback) == 18 and all('message' in row for row in feedback)
queries = ['electricity supply restoration', 'monthly water charges']
notice = 'Lakeview Municipal Utilities Maintenance Circular, 26 October 2026. Water supply to Ward 12 and Ward 14 will be suspended on 9 November 2026 for pipeline replacement. Supply is expected to resume by 18:00 on the same day. Tanker water will be provided at the Ward 12 community hall from 08:00. Electricity dues can be paid at the Civic Centre counter between 09:00 and 17:00 on working days. Leakage complaints can be registered by calling 155-300 at any time. Residents should store sufficient drinking water before the shutdown.'
qa_questions = ['On which date will water supply to Ward 12 be suspended?', 'By what time is the water supply expected to resume?', 'Where will tanker water be provided?', 'How much compensation will residents receive for the shutdown?']

print('Resident reports:', len(feedback), '| Retrieval queries:', len(queries), '| QA questions:', len(qa_questions))
display(pd.DataFrame(feedback).head(3))

Resident reports: 18 | Retrieval queries: 2 | QA questions: 4


,reference,date,message,label
0,REF-1001,01-11-2026,Reference: REF-1001. Date: 01-11-2026. Report:...,NEGATIVE
1,REF-1002,02-11-2026,Reference: REF-1002. Date: 02-11-2026. Report:...,NEGATIVE
2,REF-1003,03-11-2026,Reference: REF-1003. Date: 03-11-2026. Report:...,POSITIVE


### B. Provided BERT-family model setup

In [ ]:
# Provided setup only: the assessed prediction and evaluation code is NOT supplied.
SENTIMENT_CHECKPOINT = 'distilbert/distilbert-base-uncased-finetuned-sst-2-english'
QA_CHECKPOINT = 'distilbert/distilbert-base-cased-distilled-squad'
sentiment_tokenizer = AutoTokenizer.from_pretrained(SENTIMENT_CHECKPOINT)
sentiment_model = AutoModelForSequenceClassification.from_pretrained(SENTIMENT_CHECKPOINT)
sentiment_model.to('cpu').eval()
qa_model = pipeline('question-answering', model=QA_CHECKPOINT, device=-1)
print('Both pretrained BERT-family task models are ready on CPU.')


## Question 1 - Text processing and vector-based retrieval

Use the supplied resident reports and the two queries in `queries`. Implement the two parts described in the question paper.

- **1(a):** extracted table (request reference, date, original body, processed body) and the request references matching `water AND (leak OR leakage)`, using an inverted index.
- **1(b):** the top-5 ranked list with cosine scores for each query, the number of TF-IDF features, and a brief explanation of how query terms absent from the vocabulary affect the ranking.

In [ ]:
import pandas as pd
import os

# Let's verify if 'data.csv' exists and read it directly to avoid the upload widget blocker
if os.path.exists('data.csv'):
    df_raw = pd.read_csv('data.csv')
    print(f"Loaded {len(df_raw)} records successfully.")
    print("Sample of first 3 records:")
    for idx, row in df_raw.head(3).iterrows():
        print(f"Row {idx}: {row.to_dict()}\n")
else:
    print("data.csv not found. Please upload it or verify its path.")

data.csv not found. Please upload it or verify its path.


In [ ]:
import re
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction import text

# -----------------------------------------------------
# Question 1(a): Extraction & Text Processing
# -----------------------------------------------------
stop_words = text.ENGLISH_STOP_WORDS

extracted_records = []
for row in feedback:
    msg = row['message']

    # RegEx matching for reference, date, and original body after "Report: "
    ref_match = re.search(r"Reference:\s*(REF-\d+)", msg)
    date_match = re.search(r"Date:\s*(\d{2}-\d{2}-\d{4})", msg)
    body_match = re.search(r"Report:\s*(.*)", msg)

    ref = ref_match.group(1) if ref_match else ""
    dt = date_match.group(1) if date_match else ""
    orig_body = body_match.group(1) if body_match else ""

    # Normalise body: lowercasing, removing punctuation & digits
    clean_body = orig_body.lower()
    clean_body = re.sub(r'[^a-z\s]', '', clean_body) # Remove digits and punctuation

    # Remove scikit-learn's English stop words (keep single letters if they are not stop words)
    tokens = [w for w in clean_body.split() if w and w not in stop_words]
    proc_body = " ".join(tokens)

    extracted_records.append({
        'request_reference': ref,
        'date': dt,
        'original_body': orig_body,
        'processed_body': proc_body
    })

df_extracted = pd.DataFrame(extracted_records)

# Inverted index construction for Boolean search
inverted_index = {}
for idx, row in enumerate(extracted_records):
    ref = row['request_reference']
    words = set(row['processed_body'].split())
    for word in words:
        if word not in inverted_index:
            inverted_index[word] = set()
        inverted_index[word].add(ref)

# Query: water AND (leak OR leakage)
set_water = inverted_index.get('water', set())
set_leak = inverted_index.get('leak', set())
set_leakage = inverted_index.get('leakage', set())

boolean_results = set_water.intersection(set_leak.union(set_leakage))

print("=== Question 1(a): Extracted and Normalised Table ===")
pd.set_option('display.max_colwidth', None)
display(df_extracted)

print("\n=== Question 1(a): Matching References for 'water AND (leak OR leakage)' ===")
print(sorted(list(boolean_results)))

# -----------------------------------------------------
# Question 1(b): Vector-Based Retrieval (TF-IDF)
# -----------------------------------------------------
# TF-IDF representation of processed bodies using unigram and bigram features
vectorizer = TfidfVectorizer(ngram_range=(1, 2))
tfidf_matrix = vectorizer.fit_transform(df_extracted['processed_body'])
feature_names = vectorizer.get_feature_names_out()

print(f"\n=== Question 1(b) ===")
print(f"Number of distinct TF-IDF features: {len(feature_names)}")

# Process queries
for q in queries:
    # Normalise query terms same as document processing
    q_clean = q.lower()
    q_clean = re.sub(r'[^a-z\s]', '', q_clean)
    q_proc = " ".join([w for w in q_clean.split() if w and w not in stop_words])

    q_vector = vectorizer.transform([q_proc])
    scores = cosine_similarity(q_vector, tfidf_matrix).flatten()

    # Ranking documents
    ranked_indices = scores.argsort()[::-1][:5]

    rank_list = []
    for rank, idx in enumerate(ranked_indices, 1):
        rank_list.append({
            'Rank': rank,
            'Reference': df_extracted.iloc[idx]['request_reference'],
            'Score': round(scores[idx], 4),
            'Original Body': df_extracted.iloc[idx]['original_body']
        })

    print(f"\nTop-5 Ranked List for Query: '{q}'")
    display(pd.DataFrame(rank_list))

# Explanation for out-of-vocabulary terms
explanation = """
Brief explanation of OOV query terms:
Query terms absent from the vocabulary of the fitted TF-IDF Vectorizer do not have any matching features or dimensions in the document space.
When vectorizing the query, these out-of-vocabulary (OOV) terms are ignored (assigned zero weights), and thus have no contribution to the cosine similarity scores or final ranking rankings.
"""
print(explanation)

=== Question 1(a): Extracted and Normalised Table ===


,request_reference,date,original_body,processed_body
0,REF-1001,01-11-2026,There is a severe water leak from the main pipe on Oak Avenue. Please fix the leakage.,severe water leak main pipe oak avenue fix leakage
1,REF-1002,02-11-2026,The streetlights on Pine Street are not working. It is very dark and unsafe.,streetlights pine street working dark unsafe
2,REF-1003,03-11-2026,Thank you for restoring our electricity supply so quickly after the storm. Great service!,thank restoring electricity supply quickly storm great service
3,REF-1004,04-11-2026,Our water charges this month are extremely high. We believe our meter is faulty.,water charges month extremely high believe meter faulty
4,REF-1005,05-11-2026,The customer service executive at the civic center was very helpful and resolved my billing query promptly.,customer service executive civic center helpful resolved billing query promptly
5,REF-1006,06-11-2026,I would like to report water leakage from the municipal tank near the park. A lot of water is being wasted.,like report water leakage municipal tank near park lot water wasted
6,REF-1007,07-11-2026,Exceptional response time in repairing the broken water pipe in Ward 12. Much appreciated!,exceptional response time repairing broken water pipe ward appreciated
7,REF-1008,08-11-2026,"My electricity supply keeps fluctuating, which might damage our household appliances. Please inspect the transformer.",electricity supply keeps fluctuating damage household appliances inspect transformer
8,REF-1009,09-11-2026,The garbage collection this week was on time. The neighborhood is very clean. Thank you!,garbage collection week time neighborhood clean thank
9,REF-1010,10-11-2026,There is water pouring onto the road due to a leaking connection. Urgent attention is needed.,water pouring road leaking connection urgent attention needed



=== Question 1(a): Matching References for 'water AND (leak OR leakage)' ===
['REF-1001', 'REF-1006']

=== Question 1(b) ===
Number of distinct TF-IDF features: 259

Top-5 Ranked List for Query: 'electricity supply restoration'


,Rank,Reference,Score,Original Body
0,1,REF-1014,0.4840,We have been waiting for the restoration of electricity supply for over 12 hours now. No updates from support.
1,2,REF-1003,0.2932,Thank you for restoring our electricity supply so quickly after the storm. Great service!
2,3,REF-1008,0.2663,"My electricity supply keeps fluctuating, which might damage our household appliances. Please inspect the transformer."
3,4,REF-1016,0.0739,Friendly and efficient technical team resolved our electricity meter issue today. Thank you.
4,5,REF-1017,0.0000,The local drainage system is overflowing and causing a terrible smell. This is a health hazard.



Top-5 Ranked List for Query: 'monthly water charges'


,Rank,Reference,Score,Original Body
0,1,REF-1004,0.4135,Our water charges this month are extremely high. We believe our meter is faulty.
1,2,REF-1006,0.1095,I would like to report water leakage from the municipal tank near the park. A lot of water is being wasted.
2,3,REF-1010,0.0652,There is water pouring onto the road due to a leaking connection. Urgent attention is needed.
3,4,REF-1013,0.0646,The water pressure in our area has been very low for the past three days. We cannot fill our overhead tanks.
4,5,REF-1007,0.0614,Exceptional response time in repairing the broken water pipe in Ward 12. Much appreciated!



Brief explanation of OOV query terms:
Query terms absent from the vocabulary of the fitted TF-IDF Vectorizer do not have any matching features or dimensions in the document space. 
When vectorizing the query, these out-of-vocabulary (OOV) terms are ignored (assigned zero weights), and thus have no contribution to the cosine similarity scores or final ranking rankings.



## Question 2 - BERT applications and evaluation

Use the supplied pretrained models, the report bodies extracted in Question 1(a) (if extraction is incomplete, use the text after `Report:`), the gold labels, the maintenance circular in `notice` and the questions in `qa_questions`.

- **2(a):** predicted class and both class probabilities per record; confusion matrix, accuracy and macro-F1; comparison with an all-NEGATIVE majority baseline; table of misclassified records (state explicitly if none); original minimal pair and change in positive-class probability.
- **2(b):** span and score per question; final table with question, span, score, supported (Yes/No), final answer and one-line justification, using `NOT STATED IN SOURCE` for unsupported requests.

In [ ]:
import torch
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Safely load model if missing
SENTIMENT_CHECKPOINT = 'distilbert/distilbert-base-uncased-finetuned-sst-2-english'
try:
    sentiment_tokenizer
    sentiment_model
except NameError:
    print("Loading sentiment models...")
    sentiment_tokenizer = AutoTokenizer.from_pretrained(SENTIMENT_CHECKPOINT)
    sentiment_model = AutoModelForSequenceClassification.from_pretrained(SENTIMENT_CHECKPOINT)
    sentiment_model.to('cpu').eval()

# Extract gold labels from the original dataset
gold_labels = [row['label'] for row in feedback] # 'NEGATIVE' or 'POSITIVE'

# Predict using the loaded sentiment model
predicted_classes = []
probabilities = []

for record in extracted_records:
    body = record['processed_body'] if record['processed_body'] else record['original_body']
    inputs = sentiment_tokenizer(body, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        outputs = sentiment_model(**inputs)

    # Softmax to get probabilities
    probs = torch.softmax(outputs.logits, dim=-1).flatten().tolist()

    # SST-2 label mapping: 0 -> NEGATIVE, 1 -> POSITIVE
    pred_class = "NEGATIVE" if probs[0] > probs[1] else "POSITIVE"

    predicted_classes.append(pred_class)
    probabilities.append(probs) # [prob_negative, prob_positive]

# Format results table
results_2a = []
for i, record in enumerate(extracted_records):
    results_2a.append({
        'Reference': record['request_reference'],
        'Predicted Class': predicted_classes[i],
        'Prob (NEGATIVE)': round(probabilities[i][0], 4),
        'Prob (POSITIVE)': round(probabilities[i][1], 4),
        'Gold Label': gold_labels[i]
    })

df_results_2a = pd.DataFrame(results_2a)
print("=== Question 2(a): Sentiment Predictions ===")
display(df_results_2a)

# Metrics Calculation
acc = accuracy_score(gold_labels, predicted_classes)
macro_f1 = f1_score(gold_labels, predicted_classes, average='macro')
cm = confusion_matrix(gold_labels, predicted_classes, labels=["NEGATIVE", "POSITIVE"])

# Baseline (predicts NEGATIVE for all)
baseline_preds = ["NEGATIVE"] * len(gold_labels)
baseline_acc = accuracy_score(gold_labels, baseline_preds)
baseline_f1 = f1_score(gold_labels, baseline_preds, average='macro')

print("\n=== Question 2(a): Classification Performance ===")
print(f"Model Accuracy: {acc:.4f} | Macro-F1: {macro_f1:.4f}")
print(f"Baseline Accuracy: {baseline_acc:.4f} | Baseline Macro-F1: {baseline_f1:.4f}")
print("\nConfusion Matrix (Labels: NEGATIVE, POSITIVE):")
print(cm)

# Table of incorrectly classified records
misclassified = df_results_2a[df_results_2a['Predicted Class'] != df_results_2a['Gold Label']]
print("\n=== Question 2(a): Misclassified Records ===")
if len(misclassified) > 0:
    display(misclassified)
else:
    print("There are no misclassified records.")

# Minimal pair analysis
sentence_1 = "The technical staff resolved our meter issue today."
sentence_2 = "The technical staff ignored our meter issue today."

def get_positive_prob(text):
    inputs = sentiment_tokenizer(text, return_tensors="pt", truncation=True)
    with torch.no_grad():
        outputs = sentiment_model(**inputs)
    probs = torch.softmax(outputs.logits, dim=-1).flatten().tolist()
    return probs[1]

prob1 = get_positive_prob(sentence_1)
prob2 = get_positive_prob(sentence_2)
print("\n=== Question 2(a): Minimal Pair Analysis ===")
print(f"Sentence 1: '{sentence_1}' -> Positive Probability: {prob1:.4f}")
print(f"Sentence 2: '{sentence_2}' -> Positive Probability: {prob2:.4f}")
print(f"Change in positive-class probability: {prob2 - prob1:.4f}")

Loading sentiment models...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

=== Question 2(a): Sentiment Predictions ===


,Reference,Predicted Class,Prob (NEGATIVE),Prob (POSITIVE),Gold Label
0,REF-1001,NEGATIVE,0.9980,0.0020,NEGATIVE
1,REF-1002,NEGATIVE,0.9984,0.0016,NEGATIVE
2,REF-1003,POSITIVE,0.0006,0.9994,POSITIVE
3,REF-1004,NEGATIVE,0.9984,0.0016,NEGATIVE
4,REF-1005,POSITIVE,0.0436,0.9564,POSITIVE
5,REF-1006,NEGATIVE,0.9997,0.0003,NEGATIVE
6,REF-1007,POSITIVE,0.0024,0.9976,POSITIVE
7,REF-1008,NEGATIVE,0.9987,0.0013,NEGATIVE
8,REF-1009,POSITIVE,0.0261,0.9739,POSITIVE
9,REF-1010,NEGATIVE,0.9915,0.0085,NEGATIVE



=== Question 2(a): Classification Performance ===
Model Accuracy: 1.0000 | Macro-F1: 1.0000
Baseline Accuracy: 0.5000 | Baseline Macro-F1: 0.3333

Confusion Matrix (Labels: NEGATIVE, POSITIVE):
[[9 0]
 [0 9]]

=== Question 2(a): Misclassified Records ===
There are no misclassified records.

=== Question 2(a): Minimal Pair Analysis ===
Sentence 1: 'The technical staff resolved our meter issue today.' -> Positive Probability: 0.0495
Sentence 2: 'The technical staff ignored our meter issue today.' -> Positive Probability: 0.0020
Change in positive-class probability: -0.0475


In [ ]:
from transformers import pipeline

# Safely load model if missing
QA_CHECKPOINT = 'distilbert/distilbert-base-cased-distilled-squad'
try:
    qa_model
except NameError:
    print("Loading QA pipeline...")
    qa_model = pipeline('question-answering', model=QA_CHECKPOINT, device=-1)

qa_results = []
for q in qa_questions:
    # Run extractive QA model
    res = qa_model(question=q, context=notice)
    score = res['score']
    span = res['answer']

    # Determine support based on physical ground truth context
    supported = "No"
    final_answer = "NOT STATED IN SOURCE"
    justification = ""

    if q == 'On which date will water supply to Ward 12 be suspended?':
        if "9 November 2026" in span:
            supported = "Yes"
            final_answer = span
            justification = "Explicitly stated in the circular: 'suspended on 9 November 2026'."
    elif q == 'By what time is the water supply expected to resume?':
        if "18:00" in span:
            supported = "Yes"
            final_answer = span
            justification = "Explicitly stated in the circular: 'expected to resume by 18:00'."
    elif q == 'Where will tanker water be provided?':
        if "Ward 12 community hall" in span:
            supported = "Yes"
            final_answer = span
            justification = "Explicitly stated in the circular: 'provided at the Ward 12 community hall'."
    elif q == 'How much compensation will residents receive for the shutdown?':
        supported = "No"
        final_answer = "NOT STATED IN SOURCE"
        justification = "The circular contains no mention of monetary or other compensation values."

    qa_results.append({
        'Question': q,
        'Model Span': span,
        'Score': round(score, 4),
        'Supported': supported,
        'Final Answer': final_answer,
        'Justification': justification
    })

df_qa_results = pd.DataFrame(qa_results)
print("=== Question 2(b): QA Results Evaluation ===")
display(df_qa_results)

Loading QA pipeline...


config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/261M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cpu


=== Question 2(b): QA Results Evaluation ===


,Question,Model Span,Score,Supported,Final Answer,Justification
0,On which date will water supply to Ward 12 be suspended?,9 November 2026,0.9859,Yes,9 November 2026,Explicitly stated in the circular: 'suspended on 9 November 2026'.
1,By what time is the water supply expected to resume?,18:00,0.9248,Yes,18:00,Explicitly stated in the circular: 'expected to resume by 18:00'.
2,Where will tanker water be provided?,Ward 12 community hall,0.3173,Yes,Ward 12 community hall,Explicitly stated in the circular: 'provided at the Ward 12 community hall'.
3,How much compensation will residents receive for the shutdown?,Residents should store sufficient drinking water,0.7048,No,NOT STATED IN SOURCE,The circular contains no mention of monetary or other compensation values.


## Submission check

Ensure that all assessed cells have been executed, the result tables are visible, and the explanation for 1(b) and the justifications for 2(b) are present. You can share the link to your final jupyter notebook via email.